# 03 — Data Relabel

Define and validate a sleep-stage relabeling scheme for downstream modeling and dataset preparation.

**Notebook Objectives**

- Establish a deterministic mapping from raw hypnogram labels to a reduced label set (e.g., binary).
- Confirm that remapping preserves PSG–hypnogram alignment for epoching.
- Produce a quick class/epoch summary to assess label balance and dataset viability.

**Contents**

1. Setup and imports
2. Load a sample sleep session
3. Define remapping rules and apply to hypnogram
4. Visualize relabeled hypnogram vs PSG
5. Summarize relabeled epoch counts

**Inputs**

- Local Sleep-EDF files and remapping configuration (`src/constants.py`)

**Outputs**

- Relabeled hypnogram visualizations and dataset summaries

In [ ]:
# --- Configure the Python path to allow local imports from the src under project root ---
import sys
from pathlib import Path

# add src folder parent (the Project root) to sys.path
sys.path.insert(0, str(Path.cwd().parent))
print("Added to sys.path:", str(Path.cwd().parent))

In [ ]:
# --- Local imports ---
from src import constants as const
from src import folders as fldr
from src import plotters as plot
from src import sleep_session as ss

In [ ]:
# --- Retrieve sample sleep session file pair (EDF and hypnogram)  ---
file_pairs = fldr.get_edf_file_pairs()
print(f"Retrieved {len(file_pairs)} EDF file pairs") 

# select a representative record for structural inspection.
subject_id, psg_file, hypno_file = file_pairs[0]
print(f"Selected subject ID: {subject_id}")
print(f"Sample EDF file: {psg_file}")
print(f"Sample hypnogram file: {hypno_file}")

session = ss.SleepSession(edf_path=psg_file, hypno_path=hypno_file)

In [ ]:
# --- Raw hypnogram (signal plus label) visualization ---
filter_cfg = ss.FilterConfig(enabled=False)
remap_cfg = ss.MappingConfig(
    enabled=False, 
    remap={},  # no remapping
)
sleep_data_plot = session.get_sleep_data_plot(
    channel=const.CHANNEL_SELECTION,
    filter_cfg=filter_cfg,
    remap_cfg=remap_cfg,
    title=f"Raw hypnogram for channel {const.CHANNEL_SELECTION}",
    restrict_to_hypnogram_window=True,  # show only PSG data within hypnogram window
)
plot.plot_psg_with_hypnogram_arrays(
    psg_x_data=sleep_data_plot.psg_x_data,
    psg_x_label=sleep_data_plot.psg_x_label,
    psg_y_data=sleep_data_plot.psg_y_data,
    psg_y_label=f"{const.CHANNEL_SELECTION} (uV)",  
    psg_y_scale=sleep_data_plot.psg_y_scale,
    hypno_x_data=sleep_data_plot.hypno_x_data,
    hypno_y_data=sleep_data_plot.hypno_y_data,
    hypno_y_label=sleep_data_plot.hypno_y_label,
    title=sleep_data_plot.title,
    hypno_y_tick=sleep_data_plot.hypno_y_tick_labels,
    xlim_s=(20000, 60000)
)

In [ ]:
# --- Binary classification hypnogram (signal with label) visualization ---
filter_cfg = ss.FilterConfig(enabled=False)
remap_cfg = ss.MappingConfig(
    enabled=True,  # use original hypnogram labels for this plot
    remap=const.SLEEP_STAGE_MAP,  # no remapping
)

sleep_data_plot = session.get_sleep_data_plot(
    channel=const.CHANNEL_SELECTION,
    filter_cfg=filter_cfg,
    remap_cfg=remap_cfg,
    title=f"Combined PSG and Hypnogram for channel {const.CHANNEL_SELECTION}",
    restrict_to_hypnogram_window=True,  # show only PSG data within hypnogram window
)
plot.plot_psg_with_hypnogram_arrays(
    psg_x_data=sleep_data_plot.psg_x_data,
    psg_x_label=sleep_data_plot.psg_x_label,
    psg_y_data=sleep_data_plot.psg_y_data,
    psg_y_label=f"{const.CHANNEL_SELECTION} (uV)",  # override with microvolts for EEG
    psg_y_scale=sleep_data_plot.psg_y_scale,
    hypno_x_data=sleep_data_plot.hypno_x_data,
    hypno_y_data=sleep_data_plot.hypno_y_data,
    hypno_y_label=sleep_data_plot.hypno_y_label,
    title=sleep_data_plot.title,
    hypno_y_tick=sleep_data_plot.hypno_y_tick_labels,
    xlim_s=(20000, 60000)
)

In [ ]:
# --- binary classification dataset summary ---
sleep_dataset_epochs = session.get_sleep_dataset(
    channel=const.CHANNEL_SELECTION,
    epoch_len_s=const.EPOCH_LENGTH_SECONDS,
    filter_cfg=filter_cfg,
    remap_cfg=remap_cfg,
)
print(ss.format_sleep_dataset_summary(sleep_dataset_epochs))